In [38]:
import pandas as pd

df = pd.read_csv("studentsperformance.csv")
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [26]:
print("Number of records:", df.shape[0])
print("Number of features:", df.shape[1])
print(df.dtypes)

Number of records: 1000
Number of features: 9
gender                           str
race/ethnicity                   str
parental level of education      str
lunch                            str
test preparation course          str
math score                     int64
reading score                  int64
writing score                  int64
cluster                        int32
dtype: object


In [31]:
print(df.isna().sum())

print("Duplicate rows:", df.duplicated().sum())

categorical_cols = ["gender", "race/ethnicity", "parental level of education", "lunch", "test preparation course"]

for col in categorical_cols:
    print(f"\n{col} - unique values:")
    print(df[col].unique())
    print(f"\n{col} - value counts:")
    print(df[col].value_counts())


numeric_cols = ["math score", "reading score", "writing score"]
print("\nNumeric summary:")
print(df[numeric_cols].describe())

print("\nCategorical summary:")
print(df.describe(include="object"))

gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
cluster                        0
dtype: int64
Duplicate rows: 0

gender - unique values:
<StringArray>
['female', 'male']
Length: 2, dtype: str

gender - value counts:
gender
female    518
male      482
Name: count, dtype: int64

race/ethnicity - unique values:
<StringArray>
['group B', 'group C', 'group A', 'group D', 'group E']
Length: 5, dtype: str

race/ethnicity - value counts:
race/ethnicity
group C    319
group D    262
group B    190
group E    140
group A     89
Name: count, dtype: int64

parental level of education - unique values:
<StringArray>
[ 'bachelor's degree',       'some college',    'master's degree',
 'associate's degree',        'high school',   'some high school']
Length: 6, dtype: str

parental level of

C:\Users\matty\AppData\Local\Temp\ipykernel_14212\2063247903.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df.describe(include="object"))


Assumption 1: Score columns are bounded 0-100.

In [32]:
#Assumption 1: Score columns are bounded 0-100.
print(df[(df["math score"] < 0) | (df["math score"] > 100)])
print(df[(df["reading score"] < 0) | (df["reading score"] > 100)])
print(df[(df["writing score"] < 0) | (df["writing score"] > 100)])

Empty DataFrame
Columns: [gender, race/ethnicity, parental level of education, lunch, test preparation course, math score, reading score, writing score, cluster]
Index: []
Empty DataFrame
Columns: [gender, race/ethnicity, parental level of education, lunch, test preparation course, math score, reading score, writing score, cluster]
Index: []
Empty DataFrame
Columns: [gender, race/ethnicity, parental level of education, lunch, test preparation course, math score, reading score, writing score, cluster]
Index: []


Assumption 2: Categorical columns have consistent spelling, no near duplicates (male vs Male, trailing spacces, typos)

In [33]:
for col in categorical_cols:
    print(f"\n{col}:")
    print(sorted(df[col].unique()))


gender:
['female', 'male']

race/ethnicity:
['group A', 'group B', 'group C', 'group D', 'group E']

parental level of education:
["associate's degree", "bachelor's degree", 'high school', "master's degree", 'some college', 'some high school']

lunch:
['free/reduced', 'standard']

test preparation course:
['completed', 'none']


Assumption 3: Each row represents a unique students (no duplicates of the same records beyond exactr duplicate rows already checked)

In [34]:
print("Duplicates: {}".format(df.duplicated(subset=[c for c in df.columns if c != "math score"]).sum()))

Duplicates: 4


In [36]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import pandas as pd

# Load the data if this cell is run on its own
if "df" not in globals():
    df = pd.read_csv("studentsperformance.csv")

numeric_cols = ["math score", "reading score", "writing score"]
scaler = StandardScaler()
scaled_scores = scaler.fit_transform(df[numeric_cols])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(scaled_scores)

print(df["cluster"].value_counts())

print(pd.crosstab(df["cluster"], df["test preparation course"]))

print(pd.crosstab(df["cluster"], df["test preparation course"], normalize="index"))

cluster
0    443
2    308
1    249
Name: count, dtype: int64
test preparation course  completed  none
cluster                                 
0                              154   289
1                               51   198
2                              153   155
test preparation course  completed      none
cluster                                     
0                         0.347630  0.652370
1                         0.204819  0.795181
2                         0.496753  0.503247


In [41]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Use the three score columns as features and predict test preparation course
X = df[["math score", "reading score", "writing score"]]
y = df["test preparation course"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

baseline_guess = y_train.value_counts().idxmax()
baseline_predictions = [baseline_guess] * len(y_test)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print("Model accuracy:", round(accuracy *100, 2), "%")
print("Baseline accuracy:", round(baseline_accuracy * 100, 2), "%")

Model accuracy: 62.33 %
Baseline accuracy: 64.33 %
